# 02. Clean Model Experiments and Validation (v2)
**NetSentinel Intrusion Detection System**

In this notebook, we systematically evaluate candidate models using **leak-free block-grouped cross-validation**:
1. Honest baselines: Dummy & DecisionTree
2. Feature engineering: Network payload ratios and log transforms
3. Advanced ensembles: RandomForest and HistGradientBoosting
4. Per-class probability analysis & threshold tuning to suppress False Alarm Rate (FAR)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import sys
sys.path.append('../../')

from ml.src.config import (
    CLASSES, CLEAN_TRAIN_PATH, EXPERIMENTS_LOG_PATH, NORMAL_CLASS, N_CV_FOLDS, RANDOM_SEED
)
from ml.src.features import NetworkRatioFeatureAdder, build_preprocessor, extract_features_and_target
from ml.src.train import compute_intrusion_metrics, evaluate_model_grouped_cv

print('Evaluation configured for classes:', CLASSES)


## 1. Load Clean Preprocessed Training Data
The data has had id dropped, duplicates pruned, and chronological cv_group assigned.

In [ ]:
df_train = pd.read_csv('../../data/processed/clean_train.csv')
X, y, groups = extract_features_and_target(df_train)
print('Features matrix X shape:', X.shape)
print('Target y distribution:
', y.value_counts())
print('Group assignments (0-9):', np.unique(groups))


## 2. Inspect Experiments Log
All baseline and candidate runs evaluated with 5-fold GroupKFold are tracked in experiments_log.csv.

In [ ]:
exp_log = pd.read_csv('../../ml/results/metrics/experiments_log.csv')
print(exp_log[['model_name', 'cv_macro_f1_mean', 'cv_far_mean', 'cv_attack_recall_mean', 'cv_accuracy_mean']])


## 3. Threshold Tuning for False Alarm Suppression
In production IDS, false alarms overwhelm Security Operations Center (SOC) analysts.
By adjusting the decision threshold for the Normal vs Attack boundary, we balance Macro F1 and FAR.

In [ ]:
# Plotting trade-off curve across thresholds
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

base_prep = build_preprocessor(include_ratios=False, include_log=False)
rf_model = Pipeline([('prep', base_prep), ('clf', RandomForestClassifier(n_estimators=50, max_depth=14, random_state=RANDOM_SEED, n_jobs=-1))])
rf_model.fit(X, y)

probs = rf_model.predict_proba(X)
normal_idx = list(rf_model.classes_).index(NORMAL_CLASS)
normal_probs = probs[:, normal_idx]

thresholds = np.linspace(0.2, 0.8, 13)
fars = []
recalls = []

is_normal_true = (y == NORMAL_CLASS).values
is_attack_true = ~is_normal_true

for th in thresholds:
    # Predict attack if normal probability < th
    pred_attack = normal_probs < th
    fp = np.sum(is_normal_true & pred_attack)
    tn = np.sum(is_normal_true & ~pred_attack)
    tp = np.sum(is_attack_true & pred_attack)
    fn = np.sum(is_attack_true & ~pred_attack)
    
    far = fp / (fp + tn) if (fp + tn) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    fars.append(far)
    recalls.append(rec)

plt.figure(figsize=(8, 5))
plt.plot(thresholds, fars, marker='o', label='False Alarm Rate (FAR)', color='red')
plt.plot(thresholds, recalls, marker='s', label='Attack Recall', color='green')
plt.axvline(0.5, color='gray', linestyle='--', label='Default (0.50)')
plt.axvline(0.6, color='blue', linestyle=':', label='Tuned Threshold (0.60)')
plt.title('Threshold Tuning: Attack Recall vs False Alarm Rate')
plt.xlabel('Normal Class Probability Threshold (lower means more sensitive to attacks)')
plt.ylabel('Rate')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('../../ml/results/plots/threshold_tuning_tradeoff.png')
plt.show()


## 4. Key Conclusions from Experiments
1. **HistGradientBoosting + Ratios** achieved the highest cross-validation Macro F1 (**0.5049**), followed closely by **RandomForest + Ratios** (**0.4940**).
2. **RandomForest + Ratios** achieved the lowest False Alarm Rate (**0.1777**) among non-dummy models and provides tree-level interpretability (decision paths and tree voting) required for the frontend row inspector.
3. Feature engineering (payload ratios + log transforms) consistently improved both accuracy and macro F1 over raw features.
